<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# 💻 PID pour contrôle latéral

Dans les activités précédentes, nous avons développé les outils nécessaires au déploiement d'un contrôleur PID. Nous avons utilisé la vitesse angulaire ($\omega$) comme signal de contrôle et l'orientation du Duckiebot ($\theta$) comme variable contrôlée.

Dans cet exercice, vous utiliserez vos connaissances actuelles pour écrire un contrôleur PID afin de commander la vitesse angulaire ($\omega$) de votre Duckiebot et compenser un décalage latéral initial. Cet exercice est légèrement plus complexe que le précédent car, en raison des contraintes non holonomes du robot, nous ne pouvons pas contrôler directement cette vitesse.

Votre robot devra donc utiliser son odométrie pour estimer sa position actuelle.


## Mise en œuvre d'un contrôleur PID pour la commande latérale

Implémentez la fonction `OffsetControl` dans le fichier [pid_controller.py](../packages/solution/pid_controller.py).

Le contrôleur que vous devez écrire effectue une régulation PID sur la coordonnée $y$ (dans le repère du monde) du Duckiebot. Il reçoit les entrées suivantes :

    v_ref:      linear Duckiebot speed.
    y_ref:      reference heading pose.
    y_curr:     the current estimated "y" coordinate (offset)
    delta_t:    time interval since last call.

et devra produire les résultats suivants:

    v:       linear velocity of the Duckiebot
    omega:   angular velocity of the Duckiebot


### Test unitaire

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
sys.path.append('../packages/tests/')

from unit_test import UnitTestPositionPID

from solution.pid_controller import PIDController

# Ici, nous définissons les paramètres cinématiques.
# Ce test vous donnera une idée du comportement du contrôleur que vous avez écrit précédemment.
# Vous pouvez modifier R, la valeur de base et toutes les variables PID.
# Quels changements observez-vous sur les graphiques obtenus ? Pourquoi ?
# Note: this sanity check is neither a prerequisite for completing the exercise nor an instrument for tuning your controller. 

R = 0.0318
baseline = 0.1
gain = 0.6
trim = 0.0
v_0 = 0.2
y_ref = 0.2

# unit test input R, baseline, v_0, gain, trim, PIDController
unit_test = UnitTestPositionPID(R, baseline, v_0, y_ref, gain, trim, PIDController) 
unit_test.test()


# Ajustement du PID

Nous allons développer un régulateur PID pour contrôler deux « plants » différentes.

1. Modèle cinématique : simulation de la cinématique du Duckiebot uniquement
2. Modèle dynamique : simuler la dynamique du Duckiebot, avec les forces et l'inertie.

Vous réglerez le PID dans un scénario similaire à celui du [notebook précédent](../notebooks/PID_heading_controller.ipynb):

- **Pose de départ:** $x_0,y_0,\theta_0=[0 m, 0.2 m, 0°]$
- **Vitesse linéaire (constante):** $v_0=0.22m/s$
- **Objectif (référence)**: se rendre à $y_{ref}=0.0m$

Cela revient à démarrer sur la mauvaise voie et à essayer de suivre la ligne médiane de la bonne voie.

In [ ]:
# Conditions initiales du Duckiebot

x0 = 0.0
y0 = 0.2
theta_0 = 0.0 # degrees

initial_pose =  [x0,y0,theta_0]
initial_vel =   [0.0, 0]

# Objectif:
y_ref = 0

# Modèle cinématique

Dans cette partie du cahier, vous réglerez un contrôleur PID pour piloter un modèle cinématique simulé du Duckiebot. Il s'agit du modèle le plus simple du robot ; ce sera votre première étape pour vous familiariser avec les valeurs de gain PID acceptables.

In [ ]:
import matplotlib.pyplot as plt

from utils.writer import load_gains, update_gains

def plot_xy_from_lists(xs,ys,y_ref):
    # Plot (x,y) position
    ax = plt.subplot(313)
    plt.grid('on')
    plt.plot(xs, ys)

    plt.axhline(y = y_ref, color = 'r', linestyle = '--')

    plt.xlabel('x', fontsize=12)
    plt.ylabel('y', fontsize=12)

    # Set grass background
    ax.set_facecolor('g')

    top_line=0.2
    bottom_line=-0.2

    # Draw road
    ax.axhspan(ymin=bottom_line,ymax=top_line,facecolor='k')
    plt.axhline(y=top_line,linestyle='-',color='w') # white
    plt.axhline(y=bottom_line,linestyle='-',color='w')
    plt.axhline(y=0.0,linestyle='--',color='#fefe03')

    ax.set_ylim(bottom=-0.3,top=0.3)

    plt.show()


In [ ]:
from simulators import integrate_kinematics
from solution.pid_controller import PIDController

def compute_response(initial_pose, initial_vel = [0,0], y_ref=0,dynamics=integrate_kinematics):
    
    xs, ys, omegas, e_list, angles = dynamics(
        initial_pose,initial_vel,y_ref,controller=PIDController,
        )
        
    plot_xy_from_lists(xs,ys,y_ref)

def tune_gains(kp,kd,ki):
        # write values from sliders to file GAINS.yaml
        update_gains(kp,kd,ki)

        # simulate the controller
        compute_response(initial_pose,y_ref=y_ref,dynamics=integrate_kinematics)

from ipywidgets import interact

interact(tune_gains, kp=1.0,kd=1.0,ki=1.0)

# Modèle dynamique

Pour plus de simplicité, vous pouvez simuler ici le comportement dynamique du robot sur le plan x-y sans avoir à exécuter le simulateur complet. Cela vous permet d'ajuster les gains du PID et d'observer rapidement leur effet sur la réponse du système.

Représentons graphiquement la position du Duckiebot sur le plan xy :

In [ ]:
from simulators import integrate_dynamics
from solution.pid_controller import PIDController

def tune_gains(kp,kd,ki):
        # write values from sliders to file OFFSET_GAINS.yaml
        update_gains(kp,kd,ki)

        # simulate the controller
        compute_response(initial_pose,y_ref=y_ref,dynamics=integrate_dynamics)

interact(tune_gains, kp=1.0,kd=1.0,ki=1.0)

### 💻 Tester le contrôleur en simulation

Suivez les instructions du fichier [README](../README.md) pour créer un robot virtuel, démarrer Duckiematrix, compiler et exécuter votre code, puis lancer le navigateur noVNC.

Ouvrez VNC dans votre navigateur et cliquez sur l'icône `PID_Offset` sur votre bureau. L'interface suivante s'ouvrira (cela peut prendre environ 30 secondes, voire plus, selon les performances de votre ordinateur) :

- Une instance RVIZ préconfigurée : pour visualiser les performances

- Un terminal

- Une fenêtre d'interaction avec les champs `ref` et `v_0`, et les boutons `Envoyer des commandes` et `Arrêter`.

Dans le terminal RVIZ, vous devriez voir ce que voit le robot. Rien ne devrait bouger. Vous verrez des données de débogage dans le terminal de votre ordinateur.

Pour initialiser le test de votre contrôleur, vous devrez saisir les valeurs de `ref` (en **mètres**) et de `v_0` (entre -1 et 1), puis cliquer sur « Envoyer les commandes ». Le contrôleur que vous avez conçu précédemment calculera alors la valeur de `ω` et le robot se mettra en mouvement.

Pour commençer, $ref = 0.2$ et $v_0 = 0.2$ conviennent, mais vous pouvez modifier ces paramètres en cours d'exécution et observer les performances de votre contrôleur PID. Si vous vous éloignez trop de la route, vous pouvez utiliser le joystick pour ramener manuellement votre robot sur la route.

<figure>
  <div style="text-align:center;">
  <img src="../assets/images/pid-control/pid-lateral-sim-good.png" alt="pid-lateral-sim-1" style="width: 300px;"/>
  <p>Simulation, contrôle PID latéral. 

$$v_{0} = 0.2, y_{ref} = [0.2, -0.1]$$
</p>
  </div>
</figure>

Pour tester différentes solutions, modifiez la fonction `OffsetControl` que vous avez écrite, enregistrez le fichier, recompilez et relancez le programme.  

Pour mieux comprendre le fonctionnement du système, examinez [le ROS node](../packages/pid_controller/src/pid_controller_node.py) qui subscribe aux informations d'odométrie, appelle votre fonction, puis publie les commandes de contrôle spécifiées par votre contrôleur (Comme dans le [labo ROS basics](https://github.com/IFT3345/lx-ros-basics), nous publions directement sur le sujet `joystick_cmd` qui envoie des commandes pour contrôler le robot.).

### 🚙 Testez le contrôler sur votre Duckiebot

Vous pouvez suivre une procédure similaire à celle décrite ci-dessus (et décrite dans [README](../README.md)) pour tester votre code sur votre véritable Duckiebot.

**Remarque** : nous vous conseillons de commencer à très basse vitesse avec le Duckiebot physique.


<figure>
  <div style="text-align:center;">
  <img src="../assets/images/pid-control/pid-lateral-real-good.png" alt="pid-lateral-real-1" style="width: 300px;"/>
  <p>Duckiebot, contrôle PID latéral. 
  
  $$v_{0} = 0.2, y_{ref} = [0.2, -0.2, 0.2]$$
  </p>
  </div>
</figure>

Comme précédemment, modifiez le `OffsetControl` que vous avez écrit, enregistrez le fichier, puis reconstruisez et relancez-le.

Démontrez le comportement de votre Duckiebot à l'assistant de laboratoire.